# Phoneme collator inspection

Load a few samples and inspect the phoneme targets used by MCTC:WE.

In [ ]:
import os
import sys

PROJECT_ROOT = os.getcwd()
if os.path.basename(PROJECT_ROOT) == "notebooks":
    PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from srcs.datasets.collator import PhonemeCollator
from srcs.datasets.utils import to_text
from srcs.datasets.vicocktail import load_vicocktail
from srcs.nlp.text_transform import PhonemeTransform

In [ ]:
datasets = load_vicocktail(
    split="train",
    fraction=0.001,
    val_size=0.1,
    min_word_frequency=5,
)
dataset = datasets["train"]
sample_count = min(3, len(dataset))
items = [dataset[index] for index in range(sample_count)]

print(f"Dataset samples: {len(dataset)}")
print(f"Inspected samples: {sample_count}")

In [ ]:
phoneme_transform = PhonemeTransform()
phoneme_collator = PhonemeCollator("test", phoneme_transform)
batch = phoneme_collator(items)

print("videos:", tuple(batch["videos"].shape))
print("video_lengths:", batch["video_lengths"].tolist())
print("labels:", tuple(batch["labels"].shape))
print("label_lengths:", batch["label_lengths"].tolist())
print("component order:", phoneme_transform.component_names)

for index, item in enumerate(items):
    length = int(batch["label_lengths"][index])
    token_ids = batch["labels"][index, :length]
    components = [
        [
            phoneme_transform.id2token[name][int(component_id)]
            for name, component_id in zip(
                phoneme_transform.component_names, syllable_ids
            )
        ]
        for syllable_ids in token_ids
    ]

    print(f"\nSample {index}")
    print("Text:      ", to_text(item["label"]))
    print("IDs:       ", token_ids.tolist())
    print("Components:", components)
    print("Decoded:   ", phoneme_transform.decode(token_ids))